In [1]:
# updated_fraud_notebook.py
# Uses strict calculation-based feature engineering, trains XGBoost on whole dataset,
# saves/loads model, reports F1, accuracy, predicted vs actual fraud counts.

import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

# -------------------------
# Config
# -------------------------
MODEL_DIR = Path("new_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "xgb_fraud_model1.joblib"

# Replace with your CSV path
file_path = '../Synthetic_Financial_datasets_log.csv'

# Tolerance for float comparisons in the rules
TOL = 1e-9

# -------------------------
# Feature engineering (strict) - the 2nd version you approved
# -------------------------
def feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    """
    Strict calculation-based rule features.
    - Computes orig_delta and dest_delta (bookkeeping checks)
    - Adds logAmount
    - Adds strict rule flags:
        rule_orig_inconsistent: orig_delta != 0
        rule_dest_inconsistent: dest_delta != 0
        rule_zero_origin_drain: newbalanceOrig == 0 and oldbalanceOrg == amount
        rule_zero_dest_firstload: oldbalanceDest == 0 and newbalanceDest == amount
    - Adds fraud_strict: OR of above flags
    - Adds type_mapped for model numeric input
    """
    d = df.copy()

    # Map 'type' to numeric categories only for model pipeline (keep original categories too)
    if 'type' in d.columns:
        if d['type'].dtype == 'object' or d['type'].dtype.name == 'category':
            mapping = {"CASH_IN": 1, "CASH_OUT": 2, "DEBIT": 3, "TRANSFER": 4, "PAYMENT": 5}
            # If there are extra unseen types, assign incremental integers after the mapping
            unique_types = pd.Series(d['type'].astype(str).unique())
            extras = [t for t in unique_types if t not in mapping]
            next_id = max(mapping.values()) + 1
            for t in extras:
                mapping[t] = next_id
                next_id += 1
            d['type_mapped'] = d['type'].astype(str).map(mapping).fillna(0).astype(int)
        else:
            # numeric already
            d['type_mapped'] = d['type'].astype(int)
    else:
        d['type_mapped'] = 0

    # Drop identifiers (these do not participate in the calculation-based rules)
    d.drop(columns=["nameOrig", "nameDest"], inplace=True, errors='ignore')

    # Ensure numeric conversion (coerce errors to 0.0 to avoid NaN propagation)
    for c in ['amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest']:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors='coerce').fillna(0.0)
        else:
            # if any of the expected cols missing, create as zeros to avoid key errors later
            d[c] = 0.0

    # Bookkeeping deltas
    d['orig_delta'] = d['oldbalanceOrg'] - d['newbalanceOrig'] - d['amount']
    d['dest_delta'] = d['newbalanceDest'] - d['oldbalanceDest'] - d['amount']

    # Log amount
    d['logAmount'] = np.log1p(d['amount'].clip(lower=0))

    # Ratio features (keep but not required by strict rules; useful for model fallback)
    d['origBalanceRatio'] = np.where(d['oldbalanceOrg'] != 0, d['amount'] / d['oldbalanceOrg'], 0.0)
    d['destBalanceRatio'] = np.where(d['oldbalanceDest'] != 0, d['amount'] / d['oldbalanceDest'], 0.0)

    # Zero-balance flags
    d['origZeroBalance'] = (d['oldbalanceOrg'] == 0).astype(int)
    d['destZeroBalance'] = (d['oldbalanceDest'] == 0).astype(int)

    # Strict rule flags (use a small tolerance for float comparisons)
    tol = TOL
    d['rule_orig_inconsistent'] = (d['orig_delta'].abs() > tol).astype(int)
    d['rule_dest_inconsistent'] = (d['dest_delta'].abs() > tol).astype(int)
    d['rule_zero_origin_drain'] = ((d['newbalanceOrig'].abs() <= tol) & (d['oldbalanceOrg'].sub(d['amount']).abs() <= tol)).astype(int)
    d['rule_zero_dest_firstload'] = ((d['oldbalanceDest'].abs() <= tol) & (d['newbalanceDest'].sub(d['amount']).abs() <= tol)).astype(int)

    # Combined strict fraud flag: any of the above
    d['fraud_strict'] = ((d['rule_orig_inconsistent'] == 1) |
                         (d['rule_dest_inconsistent'] == 1) |
                         (d['rule_zero_origin_drain'] == 1) |
                         (d['rule_zero_dest_firstload'] == 1)).astype(int)

    return d

# -------------------------
# Load data
# -------------------------
print("Loading dataset from:", file_path)
df = pd.read_csv(file_path, low_memory=False)
print("Initial shape:", df.shape)

# If isFlaggedFraud exists remove it per previous request
if 'isFlaggedFraud' in df.columns:
    df = df.drop(columns=['isFlaggedFraud'])

# Basic EDA (kept minimal)
print("\nColumns:", df.columns.tolist())
print("Head:")
display(df.head())

# -------------------------
# Apply feature engineering
# -------------------------
print("\nApplying strict feature engineering ...")
df_eng = feature_engineering(df)

# Keep track of how many are covered by strict rules
rule_count = int(df_eng['fraud_strict'].sum())
print(f"Transactions flagged by strict rules: {rule_count} / {len(df_eng)}")

# -------------------------
# Build model input (include rules as features instead of forcing)
# -------------------------
model_features = [
    'amount','oldbalanceOrg','newbalanceOrig','oldbalanceDest','newbalanceDest',
    'type_mapped',
    'orig_delta','dest_delta','logAmount',
    'origBalanceRatio','destBalanceRatio',
    'origZeroBalance','destZeroBalance',
    'rule_orig_inconsistent','rule_dest_inconsistent',
    'rule_zero_origin_drain','rule_zero_dest_firstload'
]

# Ensure all exist
for f in model_features:
    if f not in df_eng.columns:
        df_eng[f] = 0.0

X = df_eng[model_features].copy()
y = df_eng['isFraud'].astype(int).values

# -------------------------
# Train or load model
# -------------------------
if MODEL_PATH.exists():
    print("\nModel file detected. Loading model from:", MODEL_PATH)
    model = joblib.load(MODEL_PATH)
else:
    print("\nNo saved model found. Training XGBoost on entire dataset...")

    neg = (y == 0).sum()
    pos = (y == 1).sum()
    scale_pos_weight = max(1.0, neg / pos) if pos > 0 else 1.0

    model = XGBClassifier(
        n_estimators=400,
        max_depth=7,
        learning_rate=0.07,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        tree_method='hist',
        scale_pos_weight=scale_pos_weight,
        use_label_encoder=False,
        eval_metric='logloss'
    )

    model.fit(X, y)
    joblib.dump(model, MODEL_PATH)
    print("Model trained on full dataset and saved to:", MODEL_PATH)

# -------------------------
# Predictions (model-only, rules are features now)
# -------------------------
print("\nGenerating predictions ...")
final_pred = model.predict(X)

# -------------------------
# Evaluation
# -------------------------
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

acc = accuracy_score(y, final_pred)
f1 = f1_score(y, final_pred)
cm = confusion_matrix(y, final_pred)
report = classification_report(y, final_pred, digits=4)

print("\n=== Evaluation (Model with rules as features) ===")
print("Accuracy:", round(acc, 6))
print("F1 Score:", round(f1, 6))
print("\nConfusion Matrix:\n", cm)
print("\nClassification Report:\n", report)
print("Predicted fraud count:", int(final_pred.sum()))
print("Actual fraud count:   ", int(y.sum()))
print(f"Transactions flagged by strict rules: {rule_count} (these were forced to fraud by rules)")

# -------------------------
# Save metadata
# -------------------------
meta = {
    'model_path': str(MODEL_PATH),
    'features_used': model_features,
    'rule_tol': TOL,
    'rule_columns': ['rule_orig_inconsistent','rule_dest_inconsistent','rule_zero_origin_drain','rule_zero_dest_firstload','fraud_strict']
}
with open(MODEL_DIR / 'metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

print("\nModel + metadata saved to:", MODEL_DIR)


Loading dataset from: ../Synthetic_Financial_datasets_log.csv
Initial shape: (6362620, 11)

Columns: ['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud']
Head:


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0



Applying strict feature engineering ...
Transactions flagged by strict rules: 6212581 / 6362620

No saved model found. Training XGBoost on entire dataset...


/Users/zeeshan.ahmad/My Github/AI-Finanacial-Fraud-Detection---EDA/.venv/lib/python3.13/site-packages/xgboost/training.py:183: UserWarning: [13:51:51] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Model trained on full dataset and saved to: new_models/xgb_fraud_model1.joblib

Generating predictions ...

=== Evaluation (Model with rules as features) ===
Accuracy: 0.999994
F1 Score: 0.997571

Confusion Matrix:
 [[6354367      40]
 [      0    8213]]

Classification Report:
               precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000   6354407
           1     0.9952    1.0000    0.9976      8213

    accuracy                         1.0000   6362620
   macro avg     0.9976    1.0000    0.9988   6362620
weighted avg     1.0000    1.0000    1.0000   6362620

Predicted fraud count: 8253
Actual fraud count:    8213
Transactions flagged by strict rules: 6212581 (these were forced to fraud by rules)

Model + metadata saved to: new_models
